In [ ]:
#Calculate order value

In [1]:
import pandas as pd
import numpy as np

In [3]:
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
from getpass import getpass

# Load the .env file
load_dotenv()

# Read database credentials
user = os.getenv("DB_USER","root")
password = os.getenv("Charan@123")
host = os.getenv("DB_HOST","localhost")
database = os.getenv("DB_NAME","cart2insights")
port = int(os.getenv("DB_PORT","3306"))

# Check whether the password is availble
if not password:
    password = getpass(
    "Enter your MYSQL password: ")

#Create the MYSQL connection URL
connection_url = URL.create(
    drivername="mysql+pymysql",
    username='root',
    password='Charan@123',
    host='127.0.0.1',
    port=3306,
    database='cart2insights'
)

#create MYSQL connection

engine = create_engine(connection_url)

#Test the connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT DATABASE()"))
        connected_database = result.scalar()
        print("Database connection successful!")
        print("Connected database:", connected_database)
except Exception as error:
    print("Database connection failed!")
    print("Error details:", error)
finally:
    engine.dispose()

Enter your MYSQL password:  ········


Database connection successful!
Connected database: cart2insights


In [ ]:
# Importing Data

In [18]:
customers = pd.read_sql("SELECT * FROM customers", engine)

In [21]:
orders = pd.read_sql("SELECT * FROM orders", engine)


In [22]:
order_items = pd.read_sql("SELECT * FROM order_items", engine)


In [23]:
products = pd.read_sql("SELECT * FROM products", engine)


In [24]:
payments = pd.read_sql("SELECT * FROM payments", engine)


In [25]:
reviews = pd.read_sql("SELECT * FROM reviews", engine)


In [26]:
sellers = pd.read_sql("SELECT * FROM sellers", engine)

In [ ]:
# Verify imported data

In [29]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Sellers:", sellers.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Products: (32951, 9)
Payments: (103886, 5)
Reviews: (99224, 7)
Sellers: (3095, 4)


In [ ]:
# Calculate order value - Item Value = Price + Freight

In [30]:
order_items["item_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

In [31]:
order_items["item_value"]

0          72.19
1         259.83
2         216.87
3          25.78
4         218.04
           ...  
112645    343.40
112646    386.53
112647    116.85
112648     64.71
112649     55.79
Name: item_value, Length: 112650, dtype: float64

In [32]:
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04
...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71


In [33]:
# Aggregate it to the order level: This gives one row per order_id.

order_totals = (
    order_items.groupby("order_id")
    .agg(
        merchandise_value=("price", "sum"),
        freight_total=("freight_value", "sum"),
        order_value=("item_value", "sum")
    )
    .reset_index()
)

In [34]:
order_totals

,order_id,merchandise_value,freight_total,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99,43.41,343.40
98662,fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,386.53
98663,fffce4705a9662cd70adb13d4a31832d,99.90,16.95,116.85
98664,fffe18544ffabc95dfada21779c9644f,55.99,8.72,64.71


In [35]:
#Calculate delivery days - This measures elapsed days from purchase to customer delivery.

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [36]:
orders["delivery_days"]

0         7.614421
1        16.216181
2         7.948437
3         6.147269
4        25.114352
           ...    
99436    17.374699
99437     9.420243
99438     4.801690
99439     1.956030
99440     5.038981
Name: delivery_days, Length: 99441, dtype: float64

In [37]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.614421
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.216181
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.948437
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.147269
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.114352
...,...,...,...,...,...,...,...,...,...
99436,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,17.374699
99437,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,9.420243
99438,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,4.801690
99439,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25,1.956030


In [38]:
#Calculate delivery delay

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

In [39]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.614421,-8.011250
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.216181,-2.330278
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.948437,-13.444954
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.147269,-5.435660
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.114352,-15.303808
...,...,...,...,...,...,...,...,...,...,...
99436,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,17.374699,-7.043981
99437,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,9.420243,-8.144502
99438,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,4.801690,-12.484468
99439,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25,1.956030,-8.083565


In [40]:
#Calculate delivery_performance

orders["delivery_performance"] = np.select(
    [
        orders["delivery_delay_days"].notna()
        & (orders["delivery_delay_days"] > 0),

        orders["delivery_delay_days"].notna()
        & (orders["delivery_delay_days"] <= 0)
    ],
    [
        "Delayed",
        "On Time or Early"
    ],
    default="Unknown"
)

In [41]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_performance
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.614421,-8.011250,On Time or Early
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.216181,-2.330278,On Time or Early
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.948437,-13.444954,On Time or Early
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.147269,-5.435660,On Time or Early
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.114352,-15.303808,On Time or Early
...,...,...,...,...,...,...,...,...,...,...,...
99436,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,17.374699,-7.043981,On Time or Early
99437,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,9.420243,-8.144502,On Time or Early
99438,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,4.801690,-12.484468,On Time or Early
99439,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25,1.956030,-8.083565,On Time or Early


In [42]:
#Calculate customer order count

customer_orders = (
    orders.merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")
    .agg(
        customer_order_count=("order_id", "nunique")
    )
    .reset_index()
)

In [45]:
customer_orders.head(60)

,customer_unique_id,customer_order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1
5,0004bd2a26a76fe21f786e4fbd80607f,1
6,00050ab1314c0e55a6ca13cf7181fecf,1
7,00053a61a98854899e70ed204dd4bafe,1
8,0005e1862207bf6ccc02e4228effd9a0,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1


In [ ]:
# Calculate repeat customer status

In [46]:


customer_orders["customer_type"] = np.where(
    customer_orders["customer_order_count"] > 1,
    "Repeat",
    "One-order customer"
)

In [48]:
customer_orders.head(60)

,customer_unique_id,customer_order_count,customer_type
0,0000366f3b9a7992bf8c76cfdf3221e2,1,One-order customer
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,One-order customer
2,0000f46a3911fa3c0805444483337064,1,One-order customer
3,0000f6ccb0745a6a4b88665a16c9f078,1,One-order customer
4,0004aac84e0df4da2b147fca70cf8255,1,One-order customer
5,0004bd2a26a76fe21f786e4fbd80607f,1,One-order customer
6,00050ab1314c0e55a6ca13cf7181fecf,1,One-order customer
7,00053a61a98854899e70ed204dd4bafe,1,One-order customer
8,0005e1862207bf6ccc02e4228effd9a0,1,One-order customer
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,One-order customer


In [ ]:
# To Calculate customer spending

In [51]:
# Step 1: Calculate order totals
order_totals = (
    order_items.groupby('order_id')['price']
    .sum()
    .reset_index(name='order_value')
)

# Step 2: Join order totals with orders
customer_orders = orders.merge(
    order_totals,
    on='order_id'
)

# Step 3: Join with customers
customer_orders = customer_orders.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id'
)

# Step 4: Calculate total spending by unique customer
customer_spending = (
    customer_orders.groupby('customer_unique_id')['order_value']
    .sum()
    .reset_index(name='total_spending')
)

In [52]:
order_totals


,order_id,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90
...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99
98662,fffcd46ef2263f404302a634eb57f7eb,350.00
98663,fffce4705a9662cd70adb13d4a31832d,99.90
98664,fffe18544ffabc95dfada21779c9644f,55.99


In [53]:
customer_orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_performance,order_value,customer_unique_id
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.614421,-8.011250,On Time or Early,58.90,871766c5855e863f6eccc05f988b23cb
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.216181,-2.330278,On Time or Early,239.90,eb28e67c4c0b83846050ddfb8a35d051
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.948437,-13.444954,On Time or Early,199.00,3818d81c6709e39d06b2738a8d3a2474
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.147269,-5.435660,On Time or Early,12.99,af861d436cfc08b2c2ddefd0ba074622
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.114352,-15.303808,On Time or Early,199.90,64b576fb70d441e8f1b2d7d446e483c5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,17.374699,-7.043981,On Time or Early,299.99,0c9aeda10a71f369396d0c04dce13a64
98662,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,9.420243,-8.144502,On Time or Early,350.00,0da9fe112eae0c74d3ba1fe16de0988b
98663,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,4.801690,-12.484468,On Time or Early,99.90,cd79b407828f02fdbba457111c38e4c4
98664,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25,1.956030,-8.083565,On Time or Early,55.99,eb803377c9315b564bdedad672039306


In [54]:
customer_spending


,customer_unique_id,total_spending
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90
2,0000f46a3911fa3c0805444483337064,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,25.99
4,0004aac84e0df4da2b147fca70cf8255,180.00
...,...,...
95415,fffcf5a5ff07b0908bd4e2dbc735a684,1570.00
95416,fffea47cd6d3cc0a88bd621562a9d061,64.89
95417,ffff371b4d645b6ecea244b27531430a,89.90
95418,ffff5962728ec6157033ef9805bacc48,115.00


In [55]:
# Calculate revenue by seller_id
seller_revenue = (
    order_items.groupby('seller_id')['price']
    .sum()
    .reset_index(name='total_revenue')
)

# Sort by highest revenue
seller_revenue = seller_revenue.sort_values(
    'total_revenue',
    ascending=False
)

print(seller_revenue.head())

                             seller_id  total_revenue
857   4869f7a5dfa277a7dca6462dcf3b52b2      229472.63
1013  53243585a1d6dc2643021fd1853d8905      222776.05
881   4a3ca9315b744ce9f8e9374361493884      200472.92
3024  fa1c13f2614d7b5c4749cbc52fecda94      194042.03
1535  7c67e1448b00f6e969d365cea6b010ab      187923.89
